<a target="_blank" href="https://colab.research.google.com/drive/1TpnI0gWV0LGcfnNCOZ_Fr53pUOWsGopB?usp=sharing">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# UNIX/Linux Networking mit C++ – Einführung in die Socket-Programmierung

Dieses Notebook erklärt die grundlegende Socket-Programmierung unter Unix/Linux mithilfe von C++. Wir implementieren einen einfachen **TCP-Server** und einen **TCP-Client**.

## Grundlagen: Was ist ein Socket?

Ein **Socket** ist ein vom Betriebssystem bereitgestelltes Objekt, das als Kommunikationsendpunkt dient. Ein Programm verwendet Sockets, um Daten mit anderen Programmen auszutauschen. Das andere Programm kann sich dabei auf demselben Computer (Interprozesskommunikation) oder einem anderen, via Netzwerk erreichbaren Computer befinden. Die Kommunikation über Sockets erfolgt in der Regel bidirektional, das heißt, über das Socket können Daten sowohl empfangen als auch gesendet werden. (siehe https://de.wikipedia.org/wiki/Socket)



### Wichtige Systemaufrufe:

| Funktion | Zweck |
|---------|-------|
| `socket()` | Erzeugt einen Socket |
| `bind()`   | Weist dem Socket eine IP/Port-Kombination zu |
| `listen()` | Macht den Socket zum Server, der Verbindungen akzeptieren kann |
| `accept()` | Wartet auf eingehende Verbindung |
| `connect()` | Verbindet Client mit Server |
| `send()` / `recv()` | Daten senden/empfangen |
| `close()` | Socket schließen |

### Wichtige Datenstrukturen:

| struct | Zweck |
|---------------|-------|
| `sockaddr_in` | Speichert IP-Adresse und Port für IPv4|
| `sockaddr` | Generische Struktur für Netzwerkadressen |
| `in_addr` | Enthält eine einzelne IPv4-Adresse |


## UNIX TCP-Server

Ein TCP-Server öffnet einen bestimmten Port und wartet auf eingehende Verbindungen. Sobald ein Client verbunden ist, kann der Server Daten empfangen und eine Antwort senden.

Im folgenden Beispiel erstellen wir einen einfachen Server, der auf Port 25565 hört und Nachrichten vom Client empfängt.

In [ ]:
%%writefile server.cpp

#include <iostream>
#include <cstring>
#include <unistd.h>

//Unix-includes für Sockets
#include <arpa/inet.h>

int main() {

    // https://man7.org/linux/man-pages/man2/socket.2.html
    // Damit wird ein Socket erzeugt.

    // Parameter:
    // AF_INET -> IPv4 wird genutzt, AF_INET6 -> IPv6
    // SOCK_STREAM -> TCP wird genutzt
    // 0 -> Betriebssystem wählt das passende Protokol

    // Return: Ein Dateideskriptor (https://de.wikipedia.org/wiki/Handle#Datei-Handle)
    int server_fd = socket(AF_INET, SOCK_STREAM, 0);

    // Falls socket() fehlschlägt, wird -1 returned
    if (server_fd < 0) {
        perror("socket failed");
        return 1;
    }

    // Struct, um eine IP und Port-Kombination zu speichern
    sockaddr_in addr{};

    addr.sin_family = AF_INET; // AF_INET -> IPv4 wird genutzt, AF_INET6 -> IPv6

    addr.sin_addr.s_addr = INADDR_ANY; // INADDR_ANY -> Jede IP kann genutzt werden, um sich zu verbinden, INADDR_LOOPBACK -> Nur der lokale Rechner kann sich verbinden (127.0.0.1)

    addr.sin_port = htons(25565); // Funktion, um eine Zahl (Port 25565 in diesem Fall) für Netzwerktransportation bereit zu machen. (https://www.gnu.org/software/libc/manual/html_node/Byte-Order.html)

    // https://man7.org/linux/man-pages/man2/bind.2.html
    // Damit wird ein Socket mit einer lokalen Adresse (IP + Port) verknüpft.

    // Parameter:
    // server_fd -> Unser Socket
    // (sockaddr*)&addr -> Unsere Kombination von IP-Adresse und Port (müssen wir als Pointer übergeben)
    // sizeof(addr) -> Größe in Bytes unserer Kombination von IP-Adresse und Port

    // Return: 0 -> Erfolgreich, -1 -> Fehler
    if (bind(server_fd, (sockaddr*)&addr, sizeof(addr)) < 0) {
        perror("bind failed");
        return 1;
    }

    // https://man7.org/linux/man-pages/man2/listen.2.html
    // Der Socket wird zu einem passiven Socket, der Verbindungen akzeptieren kann

    // Parameter:
    // server_fd -> Unser Socket
    // 1 -> Maximale Anzahl wartender Verbindungen

    // Return: 0 -> Erfolgreich, -1 -> Fehler
    if (listen(server_fd, 1) < 0) {
        perror("listen failed");
        return 1;
    }

    std::cout << "Server läuft auf Port 25565 ...\n";

    // Struct, um die IP und Port-Kombination des Clients zu speichern
    sockaddr_in client_addr{};

    // Größe in Bytes des client_addr structs
    socklen_t client_len = sizeof(client_addr);


    // https://man7.org/linux/man-pages/man2/accept.2.html
    // Akzeptziert eine eingehende Verbindung zu unserem Socket
    // Standardgemäß blockiert accept(), also wartet bis eine Verbindung eingeht

    // Parameter:
    // server_fd -> Unser Socket
    // (sockaddr*)&client_addr -> Speichert die IP-Adresse und Port-Kombination des Clients in unseren sockaddr_in struct
    // &client_len -> Größe in Bytes der Kombination von IP-Adresse und Port des Clients

    // Return: Ein neuer Socket, der nur für die Clientverbindung zuständig ist, -1 -> Fehler
    int client_fd = accept(server_fd, (sockaddr*)&client_addr, &client_len);

    if (client_fd < 0) {
        perror("accept failed");
        return 1;
    }

    char buffer[1024];

    // https://man7.org/linux/man-pages/man2/read.2.html
    // Jetzt können wir Daten von dem Socket lesen, als wäre es eine Datei:
    int n = read(client_fd, buffer, sizeof(buffer));

    std::cout << "Client sagt: " << std::string(buffer, n) << "\n"; // Empfangene Daten vom Client ausgeben

    std::string reply = "Hallo vom Server!";

    // https://man7.org/linux/man-pages/man2/send.2.html
    // Sendet Nachrichten zu einem Socket (ausgehend)

    // Parameter:
    // client_fd -> Socket vom Client
    // reply.c_str() -> Nachricht, die gesendet werden soll
    // reply.size() -> Größe der Nachricht
    // 0 ->  Standardverhalten, siehe Dokumentation für mehr Optionen

    // Return: Erfolgreich -> 0, Fehler -> -1
    send(client_fd, reply.c_str(), reply.size(), 0);

    // Schließen der Sockets, da die nicht mehr gebraucht werden
    close(client_fd);
    close(server_fd);

    return 0;
}

## UNIX TCP-Client

Der TCP-Client verbindet sich mit dem Server, sendet eine Nachricht und empfängt die Antwort.

In [ ]:
%%writefile client.cpp

#include <iostream>
#include <cstring>
#include <unistd.h>

// Unix-Include für Sockets
#include <arpa/inet.h>

int main() {

    // Wie beim Server brauchen wir einen Socket:
    int sock = socket(AF_INET, SOCK_STREAM, 0);

    if (sock < 0) {
        perror("socket failed");
        return 1;
    }

    // Auch einen struct für die IP-Adresse und Port-Kombination des Zieles
    sockaddr_in addr{};
    addr.sin_family = AF_INET;
    addr.sin_port = htons(25565);

    inet_pton(AF_INET, "127.0.0.1", &addr.sin_addr); // Funktion, um die IP-Adresse für Netzwerktransportation bereit zu machen (https://linux.die.net/man/3/inet_pton)

    // https://man7.org/linux/man-pages/man2/connect.2.html
    // Versucht, eine Verbindung zum Server aufzubauen

    // Parameter:
    // sock -> Unser Socket
    // (sockaddr*)&addr -> Struct, wo die IP-Adresse und der Port des Zieles gespeichert ist
    // sizeof(addr) -> Größe in Bytes der Kombination von IP-Adresse und Port des Zieles

    // Return: Erfolgreich -> 0, Fehler -> -1
    if (connect(sock, (sockaddr*)&addr, sizeof(addr)) < 0) {
        perror("connect failed");
        return 1;
    }

    // Selbe Funktionsweise wie beim Server

    std::string msg = "Hallo Server!";
    send(sock, msg.c_str(), msg.size(), 0);

    char buffer[1024];
    int n = read(sock, buffer, sizeof(buffer));
    std::cout << "Antwort vom Server: " << std::string(buffer, n) << "\n";

    close(sock);
    return 0;

}

## Testen von Server und Client

Wir müssen jetzt unsere Programme kompilieren:

In [ ]:
!g++ server.cpp -o server
!g++ client.cpp -o client

Damit können wir den Server starten:

In [ ]:
!./server

Sobald der Server läuft, können wir in Colab ein neues Terminal starten und mit `./client` den Client starten. Der Client sollte den Server erreichen können und gegenseitig sich Nachrichten senden und empfangen.

## HTTP Server

Mit dem Wissen, wie man einen Server unter Unix realisiert, können wir jetzt einen einfachen HTTP-Server schreiben.

In [ ]:
%%writefile httpserver.cpp

#include <iostream>
#include <unistd.h>
#include <string.h>
#include <arpa/inet.h>

#define MAX_NUMBER_OF_CLIENTS 5

int main() {

    int socketfd = socket(AF_INET, SOCK_STREAM, 0);

    if (socketfd < 0) {
        perror("Socket() failed");
        return -1;
    }

    sockaddr_in addr{};
    addr.sin_family = AF_INET;
    addr.sin_port = htons(5555); // Normalerweise laufen HTTP-Server auf Port 80 bzw. 8080
    addr.sin_addr.s_addr = INADDR_ANY;

    if (bind(socketfd, (sockaddr*)&addr, sizeof(addr)) < 0) {
        perror("Bind() failed");
        close(socketfd);
        return -1;
    }

    if (listen(socketfd, MAX_NUMBER_OF_CLIENTS) < 0) {
        perror("Listen() failed");
        close(socketfd);
        return -1;
    }

    std::cout << "Server is listening on port 5555..." << std::endl;

    sockaddr_in clientAddr{};
    socklen_t clientAddrLen = sizeof(clientAddr);

    while(true) {

        int clientSocket = accept(socketfd, (sockaddr*)&clientAddr, &clientAddrLen);
        if (clientSocket < 0) {
            perror("Accept() failed");
            continue;
        }

        // Puffer und Funktion, um die IP Adresse des Clients zu speichern
        char ip[32];
        inet_ntop(AF_INET,&clientAddr.sin_addr,ip,32);

        std::cout << "Client with IP: " <<  ip << ":" << ntohs(clientAddr.sin_port) << " connected. Sending Hello World HTTP..." << std::endl;

        // HTTP Antwort, um den Text "Hello World!" zu übertragen
        const char* msg = "HTTP/1.1 200 OK\r\n"
                          "Content-Type: text/plain\r\n"
                          "Content-Length: 27\r\n"
                          "\r\n"
                          "Hello World from Server! \r\n";

        send(clientSocket, msg, strlen(msg), 0);
        close(clientSocket);

    }


}

Kompilieren...:

In [ ]:
!g++ httpserver.cpp -o httpserver

... und starten:

In [ ]:
!./httpserver

Nun können wir Testen, ob der Server funktioniert, in dem wir im Colab Terminal folgenden Befehl ausführen:

`curl http://localhost:5555`

Man kann auch den Befehl mehrmals ausführen und sehen, wie sich der **Client-Port** (temporärer Port) nach jeder Anfrage ändert.

Mit `curl -D - localhost:5555` kann man sich die Rohdaten vom Server ausgeben.

## Chat-Software

Zum Schluss werden wir eine Chat-Software programmieren, die auch auf einer Client/Server-Architektur basiert.

Der Code ist sehr ähnlich zum TCP-Server/Client, müssen aber jetzt Gebrauch von `std::thread` machen, damit der Server/Client Nachrichten gleichzeitig senden und empfangen kann.

### Chat-Server

In [ ]:
%%writefile chatserver.cpp

#include <iostream>
#include <unistd.h>
#include <string.h>
#include <thread>
#include <atomic>
#include <arpa/inet.h>

#define PORT 10005
#define MAX_NUMBER_OF_CLIENTS 1

int serverfd;
int clientfd;


void readFromClient(int clientfd) {

    char buffer[1024];

    while(true) {

        int bytesRead = read(clientfd, buffer, sizeof(buffer) - 1);

         if(bytesRead <= 0) {

            std::cout << "Client disconnected. Shutting down Server..." << std::endl;
            close(serverfd);
            close(clientfd);
            exit(0);

        }

        buffer[bytesRead] = '\0';
        std::cout << "Client: " << buffer << std::endl;
        std::cout.flush();


    }

}

void writeToClient(int clientfd) {

    std::string message;

    while(true) {

        std::getline(std::cin, message);

        if(message == "!exit" || message == "!quit") {
          std::cout << "Closing Server. Goodbye!" << std::endl;
          close(serverfd);
          close(clientfd);
          exit(0);
        }

        send(clientfd, message.c_str(), message.length(), 0);

    }

}



int main() {

    serverfd = socket(AF_INET, SOCK_STREAM, 0);

    if(serverfd < 0) {
        perror("Socket creation failed");
        return -1;
    }

    sockaddr_in addr{};
    addr.sin_family = AF_INET;
    addr.sin_addr.s_addr = INADDR_ANY;
    addr.sin_port = htons(PORT);

    if(bind(serverfd, (sockaddr*)&addr, sizeof(addr)) < 0) {
        perror("Bind failed");
        close(serverfd);
        return -1;
    }

    if(listen(serverfd, MAX_NUMBER_OF_CLIENTS) < 0) {
        perror("Listen failed");
        close(serverfd);
        return -1;
    }

    std::cout << "Server is listening on port " << PORT << "..." << std::endl;
    std::cout << "Waiting for a client to connect..." << std::endl;

    sockaddr_in clientAddr{};
    socklen_t clientAddrLen = sizeof(clientAddr);

    clientfd = accept(serverfd, (sockaddr*)&clientAddr, &clientAddrLen);

    if(clientfd < 0) {
        perror("Accept failed");
        close(serverfd);
        return -1;
    }

    char ip[32];
    inet_ntop(AF_INET,&clientAddr.sin_addr,ip,32);
    std::cout << "Client with IP: " << ip << ":" << ntohs(clientAddr.sin_port) << " connected. " << std::endl;

    // Zwei Threads, die unsere read und write Funktionen ausführen, um Senden und Schreiben gleichzeitig zu ermöglichen
    std::thread reader(readFromClient, clientfd);
    std::thread writer(writeToClient, clientfd);

    // Warten, damit dass Programm nicht frühzeitig terminiert.
    reader.join();
    writer.join();

    close(serverfd);
    close(clientfd);

    return 0;

}



### Chat Client

In [ ]:
%%writefile chatclient.cpp

#include <iostream>
#include <unistd.h>
#include <string.h>
#include <thread>
#include <atomic>
#include <arpa/inet.h>

#define PORT 10005
#define MAX_NUMBER_OF_CLIENTS 1

int serverfd;


void readFromServer(int serverfd) {

    char buffer[1024];

    while(true) {

        int bytesRead = read(serverfd, buffer, sizeof(buffer) - 1);

         if(bytesRead <= 0) {

            std::cout << "Server closed. Shutting down Client..." << std::endl;
            close(serverfd);
            exit(0);

        }

        buffer[bytesRead] = '\0';
        std::cout << "Server: " << buffer << std::endl;
        std::cout.flush();

    }

}

void writeToServer(int serverfd) {

    std::string message;

    while(true) {

        std::getline(std::cin, message);

        if(message == "!exit" || message == "!quit") {
          std::cout << "Closing connection. Goodbye!" << std::endl;
          close(serverfd);
          exit(0);
        }

        send(serverfd, message.c_str(), message.length(), 0);

    }

}



int main() {

    serverfd = socket(AF_INET, SOCK_STREAM, 0);

    if(serverfd < 0) {
        perror("Socket creation failed");
        return -1;
    }

    sockaddr_in addr{};
    addr.sin_family = AF_INET;
    addr.sin_port = htons(PORT);

    inet_pton(AF_INET, "127.0.0.1", &addr.sin_addr);

    if(connect(serverfd, (sockaddr*)&addr, sizeof(addr)) < 0) {
        perror("Connection to server failed");
        close(serverfd);
        return -1;
    }

    std::cout << "Connected to the server on port " << PORT << "..." << std::endl;

    std::thread reader(readFromServer, serverfd);
    std::thread writer(writeToServer, serverfd);

    reader.join();
    writer.join();

    return 0;


}

Kompilieren:

In [ ]:
!g++ chatserver.cpp -o chatserver
!g++ chatclient.cpp -o chatclient

Zuerst starten wir den Server:

In [ ]:
!./chatserver

Jetzt können wir im Colab-Terminal den `./chatclient` starten und Nachrichten empfangen und schicken. Mit `!exit` oder `!quit` im Client oder Server kann man den Chat beenden.

## Übung:

Programmieren Sie einen TCP Server und Client.
<br></br>
Server:
- Wartet, bis eine Nachricht vom Client kommt
- Kommt eine Nachricht vom Client, wird diese über die Konsole ausgegeben
- Bei der Nachricht "!stopServer", soll der Server beendet werden. Achten Sie dabei, alle Sockets wieder freizugeben
<br></br>
Client:
- Verbindet sich mit dem Server. Falls der Server nicht erreichbar ist, soll eine Fehlermeldung ausgegeben und der Client beendet werden
- Die Nachricht soll vom Nutzer frei wählbar sein, also von der Konsole eingelesen werden
- Mit der Eingabe "!stopClient" wird der Client beendet
<br></br>
Ports sind frei wählbar. Starten Sie den Client von dem Colab-Terminal.

In [ ]:
%%writefile serveruebung.cpp

//TODO: Implementieren Sie den Server.

In [ ]:
%%writefile clientuebung.cpp

//TODO: Implementieren Sie den Client.

In [ ]:
!g++ serveruebung.cpp -o serveruebung
!g++ clientuebung.cpp -o clientuebung

!./serveruebung